### Подготовка датасета по показателю надой молока

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from statsmodels.graphics.tsaplots import plot_acf
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

from pylab import rcParams
from IPython.display import display
import math
from prophet import Prophet
pd.set_option('display.max_columns', 130)


import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter("ignore", category=InterpolationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)



In [2]:
df = pd.read_excel("../../Data cleansing/output data/Просуммированные по категориям с доп регрессорами.xlsx")
df.head(5)

,Показатель,Регион,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,2015-08,2015-09,2015-10,2015-11,2015-12,2016-01,2016-02,2016-03,2016-04,2016-05,2016-06,2016-07,2016-08,2016-09,2016-10,2016-11,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
0,Верблюды,АКМОЛИНСКАЯ ОБЛАСТЬ,0.00,0.00,0.40,0.00,0.00,0.00,0.09,1.00,0.00,0.00,0.20,0.00,0.00,0.18,0.28,0.00,0.00,0.40,0.00,0.00,0.65,0.00,0.31,1.00,0.00,0.00,0.00,0.00,0.00,2.01,0.00,1.20,0.00,0.00,2.18,0.39,0.00,0.00,1.04,0.00,0.14,2.08,0.00,0.00,0.00,0.00,0.00,0.30,0.00,0.00,0.00,0.66,0.00,0.33,0.00,0.9,0.00,0.00,0.00,10.08,0.00,0.0,0.00,0.0,0.00,0.54,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.36,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.40,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00
1,Верблюды,АКТЮБИНСКАЯ ОБЛАСТЬ,101.98,67.47,374.84,115.59,218.72,14.15,19.77,3.00,39.16,46.16,238.56,463.35,109.04,72.94,384.96,114.35,221.89,10.93,18.79,3.50,38.53,46.64,216.08,472.07,110.32,68.17,380.15,127.12,217.61,14.98,21.26,7.14,44.63,51.78,239.43,513.33,115.70,69.40,371.53,130.17,218.41,14.59,21.76,7.19,45.97,55.48,252.68,527.66,117.94,70.37,385.75,117.80,218.45,15.54,21.78,7.2,47.06,58.02,257.54,543.35,119.80,71.3,396.90,121.8,228.10,15.90,21.9,7.2,63.00,59.90,260.20,552.10,121.60,71.8,398.40,128.50,228.80,13.10,22.50,7.20,63.90,61.90,150.70,554.90,121.30,73.3,404.20,128.70,234.90,13.60,23.60,6.3,49.80,62.4,153.70,552.8,118.07,69.17,384.21,121.01,223.46,11.76,21.39,5.52,48.18,61.39,150.87,531.79,119.50,72.60,391.0,126.1,229.10,12.2,24.10,5.7,49.87,63.00,154.3,538.40
2,Верблюды,АЛМАТИНСКАЯ ОБЛАСТЬ,1.00,0.20,51.60,25.40,0.00,61.90,76.87,95.57,16.09,0.10,10.80,46.78,15.06,13.70,131.80,9.20,17.89,130.12,2.00,0.00,4.99,2.06,4.90,44.48,15.06,23.94,41.60,19.54,3.00,15.14,18.50,14.82,14.52,12.60,9.92,43.11,20.72,2.96,15.24,0.00,1.00,20.03,3.44,7.77,117.25,1.00,0.00,20.34,21.95,3.04,17.91,26.81,9.16,18.80,13.37,0.0,17.29,15.90,4.69,23.10,10.85,9.7,42.75,0.0,6.21,21.10,4.0,2.6,28.10,4.53,79.50,93.18,23.70,11.2,16.00,2.00,3.10,19.62,2.30,4.85,11.18,2.80,9.65,36.44,10.60,11.5,26.10,7.20,12.75,11.20,11.20,27.0,25.26,17.9,21.90,12.5,16.55,16.81,22.78,12.10,12.44,6.85,39.65,17.22,8.85,27.48,0.48,29.72,18.90,17.40,12.5,16.4,11.60,16.7,15.70,24.0,7.70,6.90,5.4,12.60
3,Верблюды,АТЫРАУСКАЯ ОБЛАСТЬ,213.89,167.70,306.60,164.97,342.57,192.00,43.60,113.03,262.37,193.97,308.17,1087.33,325.10,190.60,301.10,154.84,328.50,220.30,66.10,118.20,234.90,196.51,270.92,974.18,303.30,167.41,323.54,182.34,349.10,249.40,57.57,88.40,221.21,199.63,278.80,964.02,305.22,159.62,326.87,159.50,367.20,257.50,60.01,78.10,253.27,332.00,355.05,989.86,280.08,165.05,334.51,134.85,367.97,321.22,126.22,93.7,263.10,366.66,348.54,1041.38,293.40,154.8,339.44,143.6,405.80,308.35,88.5,130.5,575.30,458.70,357.62,998.10,292.76,188.3,487.02,129.28,421.44,351.20,95.77,97.54,624.80,631.65,499.30,1179.66,286.67,189.8,513.16,171.46,402.10,432.22,121.53,126.2,637.50,651.7,482.94,1412.2,307.85,200.59,466.60,182.63,414.56,405.19,184.80,154.96,717.92,719.01,522.83,1164.77,323.81,423.76,441.9,219.1,487.69,446.9,191.43,133.5,730.34,709.68,537.6,930.28
4,Верблюды,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,0.30,0.00,0.00,1.14,6.12,4.70

In [3]:
df['Показатель'].unique()

array(['Верблюды', 'КРС', 'Лошади', 'Молоко', 'Овцы и козы', 'Птица',
       'Свиньи', 'Яйца', 'Температура', 'Поголовье: КРС',
       'Поголовье: лошади', 'Поголовье: овцы и козы', 'Поголовье: свиньи',
       'Поголовье: птица домашняя', 'Поголовье: верблюды', 'Осадки',
       'Цена: Говядина', 'Цена: Баранина', 'Цена: Молоко', 'Цена: Яйца'],
      dtype=object)

In [4]:
df_milk = df[df['Показатель'].isin(['Молоко', 'Температура', 'Осадки', 'Поголовье: КРС', 'Цена: Молоко'])]
df_milk.sample(10)

,Показатель,Регион,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,2015-08,2015-09,2015-10,2015-11,2015-12,2016-01,2016-02,2016-03,2016-04,2016-05,2016-06,2016-07,2016-08,2016-09,2016-10,2016-11,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
383,Цена: Молоко,КАРАГАНДИНСКАЯ ОБЛАСТЬ,107.500000,108.037500,109.117875,105.189632,99.404202,97.813735,97.324666,97.519315,101.030011,105.475331,107.584838,108.875856,110.508994,110.508994,113.492736,112.244316,109.887186,106.810344,108.198879,108.307078,108.956920,112.879369,115.701354,119.519498,121.312291,124.709035,121.965436,117.452715,119.449411,119.329962,118.733312,119.208245,120.281119,121.604212,124.157900,125.647795,130.673707,131.196402,129.490848,130.138303,132.220515,133.807162,137.553762,138.929300,139.623946,143.812665,147.983232,149.611047,151.256769,150.802999,151.406211,151.406211,151.103398,152.463329,155.969985,161.896845,163.353916,167.927826,171.286383,174.198251,177.508018,175.200414,180.106025,177.944753,172.072576,182.396931,180.937755,182.204319,192.225557,202.221286,206.467933,212.249035,206.518311,204.040091,204.244132,204.244132,198.116808,199.899859,200.099759,202.300856,203.110060,214.687333,214.902020,221.563983,226.659954,227.339934,228.476634,232.360737,229.804769,226.817307,228.631845,230.232268,231.613662,235.087867,240.259800,243.623437,244.110684,243.866573,248.500038,248.500038,251.979038,253.238934,252.225978,253.487108,258.303363,259.594880,260.633259,259.851359,259.591508,262.966198,262.966198,270.329251,274.113861,274.113861,274.113861,274.113861,274.113861,274.113861,274.113861,274.936202
173,Температура,ЖАМБЫЛСКАЯ ОБЛАСТЬ,-2.306452,1.871429,3.958065,14.166667,20.106452,25.026667,27.500000,23.564516,16.060000,11.183871,4.213333,1.609677,1.487097,1.458621,9.851613,13.116667,17.854839,23.923333,24.983871,23.525806,20.176667,7.132258,0.686667,1.548387,-0.545161,-2.192857,2.222581,11.123333,19.719355,23.600000,26.409677,23.361290,18.030000,11.058065,5.483333,-2.193548,-9.332258,-3.092857,9.490323,12.286667,17.203226,23.276667,26.441935,24.106452,16.783333,10.848387,-0.003333,-1.641935,0.570968,0.592857,8.725806,12.723333,17.812903,23.080000,28.277419,24.706452,17.840000,11.819355,0.446667,1.361290,-1.964516,3.465517,6.729032,13.676667,19.390323,23.553333,26.106452,24.290323,16.403333,8.903226,-1.023333,-5.861290,-5.119355,2.778571,4.696774,12.416667,20.667742,24.876667,28.016129,25.232258,19.690000,7.790323,2.106667,2.500000,1.525806,1.542857,5.712903,17.136667,19.061290,25.230000,26.748387,22.903226,19.906667,10.900000,5.010000,-5.174194,-8.522581,0.892857,8.916129,12.773333,18.232258,25.230000,27.793548,23.964516,17.480000,13.293548,8.036667,-1.093548,0.006452,-2.175862,5.535484,12.786667,17.441935,25.286667,25.758065,25.345161,16.213333,11.387097,5.546667,-2.419355
189,Поголовье: КРС,АКТЮБИНСКАЯ ОБЛАСТЬ,365282.000000,380542.000000,400374.000000,428630.000000,448776.000000,444709.000000,443079.000000,431587.000000,418742.000000,414248.000000,404850.000000,383214.000000,376713.000000,388815.000000,408878.000000,441263.000000,463160.000000,455985.000000,457802.000000,445614.000000,432793.000000,427795.000000,417042.000000,397154.000000,386563.000000,410791.000000,431638.000000,464520.000000,488399.000000,480682.000000,476739.

In [5]:
# Step 1: Pivot to wide format (each indicator becomes columns of periods)
df_wide = df_milk.pivot(index="Регион", columns="Показатель")

# Step 2: Flatten multi-level columns: ('2015-01', 'КРС') → 'КРС_2015-01'
df_wide.columns = [f"{col[1]}_{col[0]}" for col in df_wide.columns]
df_wide = df_wide.reset_index()

# Step 3: Melt: one row per region-period-indicator
df_melted = df_wide.melt(id_vars="Регион", var_name="indicator_period", value_name="value")

# Step 4: Extract 'Период' and 'Показатель' from the combined column
df_melted["Период"] = df_melted["indicator_period"].str.extract(r"_(\d{4}-\d{2})$")
df_melted["Показатель"] = df_melted["indicator_period"].str.extract(r"^(.+)_\d{4}-\d{2}")

# Step 5: Pivot again to get final modeling format: one row per region+period, one column per indicator
df_milk = df_melted.pivot_table(index=["Регион", "Период"], columns="Показатель", values="value").reset_index()
print(df_milk.groupby("Регион").size().reset_index(name="Количество строк"))
df_milk

                            Регион  Количество строк
0              АКМОЛИНСКАЯ ОБЛАСТЬ               120
1              АКТЮБИНСКАЯ ОБЛАСТЬ               120
2              АЛМАТИНСКАЯ ОБЛАСТЬ               120
3               АТЫРАУСКАЯ ОБЛАСТЬ               120
4   ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ               120
5                          ГАЛМАТЫ               120
6                          ГАСТАНА               120
7                         ГШЫМКЕНТ                79
8               ЖАМБЫЛСКАЯ ОБЛАСТЬ               120
9    ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ               120
10          КАРАГАНДИНСКАЯ ОБЛАСТЬ               120
11            КОСТАНАЙСКАЯ ОБЛАСТЬ               120
12          КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ               120
13           МАНГИСТАУСКАЯ ОБЛАСТЬ               120
14                    ОБЛАСТЬ АБАЙ                31
15                  ОБЛАСТЬ ЖЕТІСУ                31
16                  ОБЛАСТЬ ҰЛЫТАУ                31
17            ПАВЛОДАРСКАЯ ОБЛАСТЬ            

Показатель,Регион,Период,Молоко,Осадки,Поголовье: КРС,Температура,Цена: Молоко
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-01,12800.2,9.8,372560.0,-12.490323,100.700000
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-02,15349.3,9.8,399442.0,-10.192857,100.700000
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-03,25086.1,8.3,425605.0,-5.870968,100.297200
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-04,31661.4,8.8,440023.0,4.490000,100.196903
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-05,49129.8,42.8,444647.0,14.574194,96.289224
...,...,...,...,...,...,...,...
2166,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-08,38641.4,0.0,1120067.0,27.874194,222.272429
2167,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-09,35950.3,0.5,1101103.0,20.766667,226.717878
2168,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-10,32936.5,13.6,1078583.0,13.200000,229.211775
2169,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-11,30760.6,12.1,1056289.0,6.500000,236.775763


In [6]:
df_milk = df_milk[df_milk["Регион"] != 'РЕСПУБЛИКА КАЗАХСТАН']

In [7]:
df_milk

Показатель,Регион,Период,Молоко,Осадки,Поголовье: КРС,Температура,Цена: Молоко
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-01,12800.2,9.8,372560.0,-12.490323,100.700000
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-02,15349.3,9.8,399442.0,-10.192857,100.700000
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-03,25086.1,8.3,425605.0,-5.870968,100.297200
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-04,31661.4,8.8,440023.0,4.490000,100.196903
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-05,49129.8,42.8,444647.0,14.574194,96.289224
...,...,...,...,...,...,...,...
2166,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-08,38641.4,0.0,1120067.0,27.874194,222.272429
2167,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-09,35950.3,0.5,1101103.0,20.766667,226.717878
2168,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-10,32936.5,13.6,1078583.0,13.200000,229.211775
2169,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-11,30760.6,12.1,1056289.0,6.500000,236.775763


In [8]:
df_milk.to_excel("Датасет по молоку с регрессорами.xlsx", index=False)

In [9]:
df_milk = df_milk.drop(columns=['Осадки', 'Поголовье: КРС', 'Температура', 'Цена: Молоко'])
df_milk.sample(10)

Показатель,Регион,Период,Молоко
1517,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2024-11,4408.5
238,АКТЮБИНСКАЯ ОБЛАСТЬ,2024-11,12179.5
1612,МАНГИСТАУСКАЯ ОБЛАСТЬ,2022-10,0.0
1639,ОБЛАСТЬ АБАЙ,2022-06,80058.3
623,ГАЛМАТЫ,2016-12,558.5
132,АКТЮБИНСКАЯ ОБЛАСТЬ,2016-01,4940.1
1599,МАНГИСТАУСКАЯ ОБЛАСТЬ,2021-09,0.0
1051,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2016-01,3766.9
404,АТЫРАУСКАЯ ОБЛАСТЬ,2018-09,5937.2
458,АТЫРАУСКАЯ ОБЛАСТЬ,2023-03,1652.5


In [10]:
df_milk.to_excel("Датасет по молоку.xlsx", index=False)

In [11]:
# 1) приведение типов
df = df_milk.copy()
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m", errors="coerce")
df["Молоко"] = pd.to_numeric(df["Молоко"], errors="coerce")

# 2) wide-таблица: строки — месяцы, столбцы — регионы, значения — КРС
df_milk_wide = (df
        .pivot_table(index="Период", columns="Регион", values="Молоко", aggfunc="sum")  # если дублей нет — можно aggfunc="first"
        .sort_index())
# 3) Убираем лишний уровень индекса у колонок
df_milk_wide.columns.name = None

# 4) Возвращаем "Период" в строковый формат YYYY-MM
df_milk_wide = df_milk_wide.reset_index()
df_milk_wide["Период"] = df_milk_wide["Период"].dt.strftime("%Y-%m")

df_milk_wide.to_excel("Датасет по молоку wide.xlsx", index=False)
df_milk_wide

,Период,АКМОЛИНСКАЯ ОБЛАСТЬ,АКТЮБИНСКАЯ ОБЛАСТЬ,АЛМАТИНСКАЯ ОБЛАСТЬ,АТЫРАУСКАЯ ОБЛАСТЬ,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,ГАЛМАТЫ,ГАСТАНА,ГШЫМКЕНТ,ЖАМБЫЛСКАЯ ОБЛАСТЬ,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,КАРАГАНДИНСКАЯ ОБЛАСТЬ,КОСТАНАЙСКАЯ ОБЛАСТЬ,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,МАНГИСТАУСКАЯ ОБЛАСТЬ,ОБЛАСТЬ АБАЙ,ОБЛАСТЬ ЖЕТІСУ,ОБЛАСТЬ ҰЛЫТАУ,ПАВЛОДАРСКАЯ ОБЛАСТЬ,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,ТУРКЕСТАНСКАЯ ОБЛАСТЬ
0,2015-01,12800.2,4854.4,44442.1,2093.2,30300.3,315.1,34.8,NaN,14848.0,3611.2,9281.2,17605.7,5111.5,98.7,NaN,NaN,NaN,13983.1,15660.9,NaN
1,2015-02,15349.3,13789.3,38614.3,2274.2,37867.1,253.9,34.6,NaN,14639.9,6283.7,15214.9,22901.4,4922.2,113.7,NaN,NaN,NaN,17224.7,17551.2,NaN
2,2015-03,25086.1,26250.2,50253.8,3418.8,53846.5,476.0,42.7,NaN,15844.1,9171.8,31818.5,22890.2,5680.7,125.6,NaN,NaN,NaN,24506.8,20396.7,NaN
3,2015-04,31661.4,22545.1,56877.9,5912.4,65609.0,594.9,47.2,NaN,25219.9,18468.2,29900.4,43794.5,6401.1,134.7,NaN,NaN,NaN,28898.0,49419.5,NaN
4,2015-05,49129.8,51241.8,73872.6,6679.5,103252.2,272.9,55.4,NaN,37200.8,40125.2,64324.9,91658.2,7594.6,893.7,NaN,NaN,NaN,47215.8,68410.5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2024-08,22673.0,19708.6,32883.5,3263.7,25528.8,27.5,25.0,6371.8,24907.0,20199.8,18272.4,13406.8,4363.4,0.0,32329.0,20792.5,7304.7,24757.9,43225.3,38641.4
116,2024-09,22635.3,18126.4,32588.5,3272.5,21278.6,22.9,17.4,5602.1,24341.0,19579.4,23386.7,12774.7,4359.7,0.0,30870.5,25258.8,3080.6,20555.9,37806.9,35950.3
117,2024-10,18373.4,14585.2,30090.5,1500.4,19066.4,28.7,12.4,5910.7,15457.5,11174.1,13690.0,11421.4,4199.9,0.0,24860.8,18643.5,6417.0,18701.1,26249.9,32936.5
118,2024-11,15949.9,12179.5,22030.8,2262.7,19393.8,28.2,6.6,5144.2,14452.4,9415.9,12115.0,8236.4,4408.5,0.0,19961.8,15494.9,3776.3,16120.0,20416.9,30760.6
